In [60]:
# rag

In [61]:
# 2 step rag
#  1.Retrival -Embedding Model -- OPen ai or ollama
#     2. Generation - LLM or generative model -- Open ai or ollama

In [62]:
from langchain.chat_models import init_chat_model #LLM
from langchain.embeddings import init_embeddings #Embedding model
# from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

In [63]:
text = open(file="doc.txt",encoding="utf-8").read()

In [64]:
doc = Document(page_content=text, metadata={"source": "doc.txt"})

In [65]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600, chunk_overlap=100
)

In [66]:
chunks = splitter.split_documents(documents=[doc])

In [67]:
len(chunks)

8

In [68]:
chunks

[Document(metadata={'source': 'doc.txt'}, page_content='# ShopPK Customer Policy Handbook (2026 edition)\n\n## 1. Refunds\n\n### Refund window\nCustomers can request a refund within 14 days of delivery. Items must be unused and in their original packaging.\n\n### How refunds are paid\nRefunds are paid to the original payment method within 5 working days after the item reaches our warehouse in Lahore. Cash on delivery orders are refunded by bank transfer or JazzCash.\n\n### Items that cannot be refunded\nPerishable groceries, personal care items that have been opened, and gift cards cannot be refunded.\n\n## 2. Electronics'),
 Document(metadata={'source': 'doc.txt'}, page_content='## 2. Electronics\n\n### Refund window for electronics\nElectronics have a shorter 7-day refund window because they lose value quickly once opened.\n\n### Warranty instead of refund\nAfter 7 days, electronics are covered by a 1-year warranty repair instead of a refund. Customers keep the original box for any w

In [69]:
embeddings = init_embeddings(
    model="nomic-embed-text:latest",
    provider = "ollama"
)

In [70]:
vs = InMemoryVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [71]:
vs

In [72]:
llm = init_chat_model(
    model="gemma4:31b-cloud",
    model_provider="ollama"
)

In [73]:
query = "How refund are paid"

In [74]:
texts = vs.similarity_search(query, k=3)

In [75]:
texts

[Document(id='65fdbda4-577c-4cad-8432-5a92ea038223', metadata={'source': 'doc.txt'}, page_content='# ShopPK Customer Policy Handbook (2026 edition)\n\n## 1. Refunds\n\n### Refund window\nCustomers can request a refund within 14 days of delivery. Items must be unused and in their original packaging.\n\n### How refunds are paid\nRefunds are paid to the original payment method within 5 working days after the item reaches our warehouse in Lahore. Cash on delivery orders are refunded by bank transfer or JazzCash.\n\n### Items that cannot be refunded\nPerishable groceries, personal care items that have been opened, and gift cards cannot be refunded.\n\n## 2. Electronics'),
 Document(id='35d8d877-41ec-4678-95ab-2e441c6c09bc', metadata={'source': 'doc.txt'}, page_content='## 2. Electronics\n\n### Refund window for electronics\nElectronics have a shorter 7-day refund window because they lose value quickly once opened.\n\n### Warranty instead of refund\nAfter 7 days, electronics are covered by a

In [76]:
context = [d.page_content for d in texts]

# or we can write it like
# context = []
# for d in texts:
#     context.append(d.page_content)

In [77]:
ctx = "\n\n".join(context) # list to string

# or this all we can write it like this which is called generator expression
# ctx = "\n\n".join(d.page_content for d in texts)

In [78]:
ctx

'# ShopPK Customer Policy Handbook (2026 edition)\n\n## 1. Refunds\n\n### Refund window\nCustomers can request a refund within 14 days of delivery. Items must be unused and in their original packaging.\n\n### How refunds are paid\nRefunds are paid to the original payment method within 5 working days after the item reaches our warehouse in Lahore. Cash on delivery orders are refunded by bank transfer or JazzCash.\n\n### Items that cannot be refunded\nPerishable groceries, personal care items that have been opened, and gift cards cannot be refunded.\n\n## 2. Electronics\n\n## 2. Electronics\n\n### Refund window for electronics\nElectronics have a shorter 7-day refund window because they lose value quickly once opened.\n\n### Warranty instead of refund\nAfter 7 days, electronics are covered by a 1-year warranty repair instead of a refund. Customers keep the original box for any warranty claim.\n\n## 11. Contact and Support Hours\nOur support team is available Monday to Saturday, 9 am to 9

In [79]:
llm.invoke(input=f"Answer the question based on the context below:\nContext: {ctx}\n\nQuestion: {query}\n\nAnswer:").text


'Refunds are paid to the original payment method within 5 working days after the item reaches the warehouse in Lahore. For cash on delivery orders, refunds are made via bank transfer or JazzCash.'

In [84]:
prompt_template = [
    {
        "role": "system",
        "content": "You are a helpful assistant that answers questions based on the context provided."
    },
    {
        "role": "user",
        "content": f"Answer the question based on the context below:\n"
                   f"Context: {ctx}\n\nQuestion: {query}\n\nAnswer:"
    }
]

In [85]:
print(llm.invoke(input=prompt_template).text)

Refunds are paid to the original payment method within 5 working days after the item reaches the warehouse in Lahore. For cash on delivery orders, refunds are issued via bank transfer or JazzCash.
